In [0]:
%pip install yfinance


In [0]:
from datetime import datetime, timezone
import json
import requests
import time
import yfinance as yf

default_date = "2026-08-28"  # Target date
# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Schema Name")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "4. Secret Scope")
dbutils.widgets.text("secret_key", "finnhub-api-key", "5. Secret Key")
dbutils.widgets.text("api_key_direct", "", "6. Direct Finnhub API Key (Optional)")
dbutils.widgets.text("tickers", "AAPL,NVDA,MSFT,AMZN,TSLA,QQQ", "7. Tickers")
dbutils.widgets.text("start_date", default_date, "8. Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", default_date, "9. End Date (YYYY-MM-DD, inclusive)")
dbutils.widgets.text("target_date", "", "10. Target Date (Fallback if start_date empty)")
dbutils.widgets.text("target_file_count", "500", "11. Target File Count")
dbutils.widgets.text("landing_override_path", "", "12. Override Landing Path")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
direct_api_key = dbutils.widgets.get("api_key_direct").strip()
tickers = [t.strip() for t in dbutils.widgets.get("tickers").split(",")]
start_date_input = dbutils.widgets.get("start_date").strip()
end_date_input = dbutils.widgets.get("end_date").strip()
target_date_input = dbutils.widgets.get("target_date").strip()
target_file_count = int(dbutils.widgets.get("target_file_count"))
override_path = dbutils.widgets.get("landing_override_path").strip()

# Resolve date range: start_date takes precedence, falls back to target_date
start_date = start_date_input or target_date_input or default_date
end_date = end_date_input or start_date

print(f">>> Fetching News Articles for Window: {start_date} to {end_date} (inclusive) <<<")

# Resolve landing path
if override_path:
    landing_path = override_path
else:
    try:
        user_name = spark.sql("SELECT current_user()").collect()[0][0]
        if catalog == "workspace" or "gmail" in user_name:
            landing_path = f"/Workspace/Users/{user_name}/nasdaq_landing/landing/finnhub_news"
        else:
            landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"
    except Exception:
        landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"

dbutils.fs.mkdirs(landing_path)
print(f"Target Landing Path: {landing_path}")
print(f"Active News Ingestion Window: {start_date} to {end_date}")

# 1. Resolve Finnhub API Key (Direct widget or Databricks secret scope)
api_key = direct_api_key
if not api_key:
    try:
        api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    except Exception:
        api_key = ""

unique_articles = {}

# 2. Source A: Live Finnhub News API (Primary Source - Strictly Date Aligned)
if api_key:
    print(f"Fetching date-aligned real news from Finnhub API for window: {start_date} to {end_date}...")
    try:
        # General market news (strictly filtered by date window)
        general_url = f"https://finnhub.io/api/v1/news?category=general&token={api_key}"
        res_gen = requests.get(general_url, timeout=10)
        gen_count = 0
        if res_gen.status_code == 200 and isinstance(res_gen.json(), list):
            for article in res_gen.json():
                art_ts = article.get("datetime", 0)
                art_date = datetime.fromtimestamp(art_ts, tz=timezone.utc).strftime("%Y-%m-%d")
                if start_date <= art_date <= end_date:
                    art_id = str(article.get("id"))
                    if art_id and art_id not in unique_articles:
                        article["_schema_phase"] = "general_market"
                        unique_articles[art_id] = article
                        gen_count += 1
            print(f"[Finnhub] Landed {gen_count} real general market articles for window {start_date} to {end_date}.")
        time.sleep(1.1)

        # Company-specific news for target date / range
        for symbol in tickers:
            if len(unique_articles) >= target_file_count:
                break
            url = f"https://finnhub.io/api/v1/company-news?symbol={symbol}&from={start_date}&to={end_date}&token={api_key}"
            res = requests.get(url, timeout=10)
            ticker_count = 0
            if res.status_code == 200 and isinstance(res.json(), list):
                for article in res.json():
                    art_id = str(article.get("id"))
                    if art_id and art_id not in unique_articles:
                        article["_schema_phase"] = "nasdaq100_company"
                        article["index_tracker"] = "NASDAQ-100"
                        unique_articles[art_id] = article
                        ticker_count += 1
            print(f"[Finnhub] Landed {ticker_count} real company news articles for {symbol} ({start_date} to {end_date})")
            time.sleep(1.1)
    except Exception as e:
        print(f"[Finnhub Error]: {e}")

# 3. Source B: Live Real Market News via Yahoo Finance (Fallback if Finnhub produced 0 articles)
if not unique_articles:
    print(f"Finnhub returned 0 articles for {start_date} to {end_date}. Activating Yahoo Finance fallback...")
    for symbol in tickers:
        try:
            t = yf.Ticker(symbol)
            news_list = t.news or []
            ticker_count = 0
            for item in news_list:
                title = item.get("title")
                if not title:
                    continue
                pub_time = item.get("providerPublishTime", int(datetime.now(timezone.utc).timestamp()))
                pub_date = datetime.fromtimestamp(pub_time, tz=timezone.utc).strftime("%Y-%m-%d")
                if not (start_date <= pub_date <= end_date):
                    continue
                raw_uuid = item.get("uuid", "")
                art_id = abs(hash(f"{symbol}_{pub_time}_{raw_uuid}")) % (10**12)
                str_id = str(art_id)
                if str_id not in unique_articles:
                    unique_articles[str_id] = {
                        "id": art_id,
                        "datetime": int(pub_time),
                        "headline": title,
                        "summary": item.get("summary", title),
                        "related": symbol,
                        "source": item.get("publisher", "Yahoo Finance"),
                        "category": "company" if symbol != "QQQ" else "general",
                        "url": item.get("link", f"https://finance.yahoo.com/quote/{symbol}"),
                        "image": "",
                        "_schema_phase": "real_market_news",
                        "index_tracker": "NASDAQ-100",
                        "_ingested_at": datetime.now(timezone.utc).isoformat()
                    }
                    ticker_count += 1
            print(f"[yfinance fallback] Found {ticker_count} real news articles for {symbol} on {start_date}")
        except Exception as e:
            print(f"[yfinance fetch error for {symbol}]: {e}")
else:
    print(f"Primary source (Finnhub) successfully populated {len(unique_articles)} date-aligned articles. Fallback not needed.")

# 4. Write each unique article as an individual landing JSON file keyed by ArticleId
all_articles = list(unique_articles.values())[:target_file_count]
for article in all_articles:
    art_id = str(article.get("id"))
    file_name = f"{landing_path}/news_{art_id}.json"
    with open(file_name, "w") as f:
        json.dump(article, f)

print(f">>> Successfully landed {len(all_articles)} REAL date-aligned news files into: {landing_path} <<<")


In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# volume = dbutils.widgets.get("volume")
# landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"

# files = dbutils.fs.ls(landing_path)
# print(f"Total Landed JSON Files: {len(files)}")

# if files:
#     print("--- Sample Landed JSON Content ---")
#     sample_path = files[0].path
#     sample_json = spark.read.option("multiline", "true").json(sample_path)
#     display(sample_json)